# Prewise Qwen3.5-4B Multi-Adapter Training

Chọn **GPU T4 x2**. Bundle huấn luyện phải được attach dưới dạng Kaggle Dataset. Ba adapter được train riêng để giữ contract và rollback độc lập.

## 1. Cài thư viện

Chạy cell sau, sau đó **restart session một lần**.

In [ ]:
!pip install --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo
!pip install --upgrade --no-cache-dir "transformers>=5.2.0" "trl>=0.26.2" datasets accelerate peft bitsandbytes jsonschema


## 2. Tìm và validate bundle

In [ ]:
from pathlib import Path

matches = list(Path('/kaggle/input').rglob('dataset_manifest.json'))
assert len(matches) == 1, f'Expected exactly one bundle, found: {matches}'
BUNDLE_ROOT = matches[0].parent
TRAIN_SCRIPT = BUNDLE_ROOT / 'scripts' / 'train_prewise_adapters_kaggle.py'
VALIDATE_SCRIPT = BUNDLE_ROOT / 'scripts' / 'validate_prewise_training_bundle.py'
OUTPUT_ROOT = Path('/kaggle/working/prewise-adapters')
print('BUNDLE_ROOT =', BUNDLE_ROOT)


In [ ]:
!python "$VALIDATE_SCRIPT" --bundle-root "$BUNDLE_ROOT"


## 3. Message Context Adapter

In [ ]:
!torchrun --standalone --nproc_per_node=2 "$TRAIN_SCRIPT" \
  --bundle-root "$BUNDLE_ROOT" \
  --task message_context \
  --output-root "$OUTPUT_ROOT" \
  --max-seq-length 2048 \
  --epochs 1 \
  --smoke-test


## 4. Web Context Adapter

In [ ]:
!torchrun --standalone --nproc_per_node=2 "$TRAIN_SCRIPT" \
  --bundle-root "$BUNDLE_ROOT" \
  --task web_context \
  --output-root "$OUTPUT_ROOT" \
  --max-seq-length 2048 \
  --epochs 1 \
  --smoke-test


## 5. Explanation Adapter

In [ ]:
!torchrun --standalone --nproc_per_node=2 "$TRAIN_SCRIPT" \
  --bundle-root "$BUNDLE_ROOT" \
  --task explanation \
  --output-root "$OUTPUT_ROOT" \
  --max-seq-length 2048 \
  --epochs 1 \
  --smoke-test


## 6. Kiểm tra package và đóng ZIP

In [ ]:
import json
from pathlib import Path

required = [
    OUTPUT_ROOT / 'message-context-adapter/current/adapter_config.json',
    OUTPUT_ROOT / 'message-context-adapter/current/adapter_model.safetensors',
    OUTPUT_ROOT / 'web-context-adapter/current/adapter_config.json',
    OUTPUT_ROOT / 'web-context-adapter/current/adapter_model.safetensors',
    OUTPUT_ROOT / 'explanation-adapter/current/adapter_config.json',
    OUTPUT_ROOT / 'explanation-adapter/current/adapter_model.safetensors',
    OUTPUT_ROOT / 'manifest.json',
]
missing = [str(path) for path in required if not path.is_file()]
assert not missing, missing
for path in required:
    print(path.relative_to(OUTPUT_ROOT), path.stat().st_size)


In [ ]:
import shutil
archive = shutil.make_archive('/kaggle/working/prewise-adapters', 'zip', OUTPUT_ROOT)
print(archive)
